In [ ]:
import sys
sys.path.append("..")
from datetime import datetime
import torch, numpy as np

from src.environment.utils import smooth
from src.data import load_and_align_data, PAIRS, get_field

from bokeh.palettes import Category10
import bokeh.plotting as bk
bk.output_notebook()

In [ ]:
PAIRS

# Read Historical Data

In [ ]:
PAIRS_ = {
    'Bitcoin': 'XBTEUR',
    'Ethereum': 'ETHEUR',
    'Ripple': 'XRPEUR',
    'Cardano': 'ADAEUR',
    'Solana': 'SOLEUR',
}

In [ ]:
data, times = load_and_align_data(PAIRS_, interval=1)

In [ ]:
times_ = torch.tensor([t.timestamp() for t in times], dtype=torch.float64)
dt = float(times_.diff().mean().round())
print(f"dt = {dt}")

prices = torch.tensor(get_field(data, 'close')).T
volume = torch.tensor(get_field(data, 'volume')).T

history = [{
    'time'  : t,
    'prices': p,
    'volume': v
} for t, p, v in zip(times_, prices, volume)]

history = sorted(history, key=lambda x: x['time'])

len(history)

In [ ]:
from src.environment.proto_v07_discrete import MultiCurrencyEnv

base_t = 60
tau_p = torch.tensor([base_t*20, base_t*60*3, base_t*60*24, base_t*60*24*7], dtype=torch.float32)
print("tau_p:", (tau_p / (3600 * 24)).tolist(), "[days]")

env = MultiCurrencyEnv(
    N=len(data),
    C0=1_000,
    tau_p=tau_p,
    temp=1.0,  # unused now, kept for compatibility
    bankruptcy_threshold=1.0,
    transaction_eps=1e-2,
    use_dollar_volume=True,
    save_history=False,
    sell_fee=1.0,
    buy_fee=1.0,
    tax_rate=0.26,
    min_buy_dollars=10.0,
    dV_coeff=0.02,
    size_buckets=(0.50, 1.00),
    invalid_trade_penalty=0.2,
    dtype=torch.float32,
    eps=1e-8,
)
print(f"state_dim: {env.state_dim} - action_dim: {env.action_dim}")

# Create Agent

In [ ]:
from src.agent.dqn import RecurrentDQNAgent

agent = RecurrentDQNAgent(
    self_state_dim = env.state_dim,
    action_dim = env.action_dim,
    # rainbow properties
    gamma = 0.999,
    n_step = 3,
    replay_capacity = 500_000,
    batch_size = 64,
    target_update_interval = 250,
    double_dqn = True,
    # PER
    per_alpha = 0.6,
    per_beta = 0.4,
    per_beta_increment = 1e-6,
    per_eps = 1e-6,
    # C51
    num_atoms = 51,
    v_min = -10.0,
    v_max = 10.0,
    # network properties
    hidden_dims = (512,),
    hidden_dims_value = [512],
    hidden_dims_advantage = [512],
    activation = torch.LeakyReLU(negative_slope=0.02),
    dtype = torch.float32,
    device = "cpu",
)
# agent.load(f"../data/agent/rdqn_v07_discrete.ptm")

# Main Training Loop

In [ ]:
loss, rewards, info = [], [], []

In [ ]:
_loss, _rewards, _info = agent.train_on_historical(
    env, history[:int(len(history)*0.9)], n_episodes=100,
    update_interval=512, n_updates=4,
    max_steps=60000, warm_up=12000,
    lr=1e-3, optim="AdamW", init_optimizer=True,
    max_grad_norm=1.0,
)
agent.save(f"../data/agent/rdqn_v07_discrete.ptm")

loss    += _loss
rewards += _rewards
info    += _info

In [ ]:
t_all = np.concatenate([np.linspace(i,i+1,len(episode_loss), endpoint=False) for i, episode_loss in enumerate(loss)])
loss_l_all = np.concatenate([np.array([loss_dict['loss'] for loss_dict in episode_loss]) for episode_loss in loss])
loss_q_all = np.concatenate([np.array([loss_dict['q_mean'] for loss_dict in episode_loss]) for episode_loss in loss])
loss_e_all = np.concatenate([np.array([loss_dict['td_error'] for loss_dict in episode_loss]) for episode_loss in loss])

figl = bk.figure(title="Total Loss", x_axis_label="Training Iteration [Epochs]", y_axis_label="Loss", width=900, height=320)
figl.line(t_all, loss_l_all.reshape(-1), line_width=2, legend_label="Batch Loss", color=Category10[10][4])

figq = bk.figure(title="Q Average", x_axis_label="Training Iteration [Epochs]", y_axis_label="Loss", width=900, height=320)
figq.line(t_all, loss_q_all.reshape(-1), line_width=2, legend_label="Batch Loss", color=Category10[10][0])

fige = bk.figure(title="TD Errors", x_axis_label="Training Iteration [Epochs]", y_axis_label="Loss [1]", width=900, height=320)
fige.line(t_all, loss_e_all.reshape(-1), line_width=2, legend_label="Batch Loss", color=Category10[10][2])

bk.show(figl)
bk.show(figq)
bk.show(fige)

In [ ]:
rewards_ = [sum(r) for r in rewards]

fig = bk.figure(title="Total Rewards", x_axis_label="Training Iteration [Episodes]", y_axis_label="Loss", width=900, height=320)
fig.line(list(range(len(rewards_))), rewards_, line_width=2, legend_label="Total Reward / Episode", color=Category10[10][4])
fig.legend.location = "bottom_right"
bk.show(fig)

rewards_ = [sum(r) / len(r) for r in rewards]

fig = bk.figure(title="Average Rewards", x_axis_label="Training Iteration [Episodes]", y_axis_label="Reward", width=900, height=320)
fig.line(list(range(len(rewards_))), rewards_, line_width=2, legend_label="Average Reward / Episode", color=Category10[10][4])
fig.legend.location = "bottom_right"
bk.show(fig)

In [ ]:
episode = -1

ts = [datetime.fromtimestamp(item['t']) for item in info[episode]]
ps = torch.tensor([item['p'] for item in info[episode]])
Vs = torch.tensor([item['V'] for item in info[episode]])
Cs = torch.tensor([item['C'] for item in info[episode]])
vs = torch.tensor([[w * p  / item['V'] for w, p in zip(item['w'], item['p'])] for item in info[episode]])

f0 = bk.figure(title=f"Episode {episode} - Prices", x_axis_label="t", y_axis_label=r"\(p / p_{max} [1]\)", x_axis_type="datetime", width=900, height=320)
for i, name in enumerate(data):
    f0.line(ts, ps[:,i] / ps[:,i].max(), line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
    # f0.line(ts, smooth(ps[:,i] / ps[:,i].max(), [item['t'] for item in info[episode]], tau=60*3), line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f0.legend.click_policy = "hide"
bk.show(f0)

f0 = bk.figure(title=f"Episode {episode} - Portfolio Fraction", x_axis_label="t", y_axis_label="EUR", x_axis_type="datetime", width=900, height=320)
f0.line(ts, Cs / Vs, line_width=2, line_dash="dashed", legend_label="Cash", color=Category10[10][0])
for i, name in enumerate(data):
    f0.line(ts, vs[:,i], line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f0.legend.click_policy = "hide"
bk.show(f0)

f1 = bk.figure(title=f"Episode {episode} - Portfolio Value", x_axis_label="t", y_axis_label="EUR", x_axis_type="datetime", width=900, height=320)
r1 = f1.line(ts, Vs, line_width=2, legend_label="Total V")
r2 = f1.line(ts, Cs, line_width=1, line_dash="dashed", legend_label="Cash C")
f1.legend.click_policy = "hide"
bk.show(f1)

fig = bk.figure(title=f"Episode {episode} - Rewards", x_axis_label="t", y_axis_label=r"Reward \(\left(\log(\frac{V_{t+1}}{V_t})\right)\)", x_axis_type="datetime", width=900, height=320)
fig.scatter(ts, rewards[episode], size=2, color=Category10[10][4], legend_label="Reward")

returns = []
gamma = 0.999
G = 0.0
for r in reversed(rewards[episode]):
    G = r + gamma * G
    returns.insert(0, G)
fig.line(ts, returns, line_width=2, color=Category10[10][5], legend_label="Return")

hist, edges = torch.histogram(torch.tensor(rewards[episode]), bins=200, density=True)
x = (edges[:-1] + edges[1:]) / 2
figh = bk.figure(title="Reward Distribution", width=300, height=320)
figh.harea(y=x, x1=0, x2=hist, fill_color=Category10[10][4], fill_alpha=0.4)
figh.line(hist, x, line_color=Category10[10][4], line_width=2)
fig.legend.click_policy = "hide"

bk.show(bk.row(fig, figh))

In [ ]:
_actions = torch.tensor([item['action_frac'] for item in info[episode]])
ts = [j for j in range(len(_actions))]

f1 = bk.figure(title=f"Action Fraction", x_axis_label="t", y_axis_label="Buy/Sell [USD]", width=900, height=320)
f1.scatter(ts, _actions, size=3, legend_label=f"a_{i+1}", color=Category10[10][(i)%10])
f1.line(ts, _actions, line_width=1, line_dash="dashed", legend_label=f"a_{i+1}", color=Category10[10][(i)%10])
f1.legend.click_policy = "hide"
bk.show(f1)

In [ ]:
raise

# Validate

In [ ]:
start = int(len(history)*0.9)

env.save_history = True

hist_s = []
hist_r = []
hist_i = []

state = env.reset(history[start])
for elem in history[start:]:
    a = agent.act(state.to_tensor(), explore=False)
    state, reward, done, _info = env.step(a, data=elem)

    hist_s.append(state)
    hist_r.append(reward)
    hist_i.append(_info)

    if done:
        break

In [ ]:
Vs = [item['V'] for item in hist_i]
Cs = [item['C'] for item in hist_i]
ts = [datetime.fromtimestamp(item["t"]) for item in hist_i]

skip = 10
f1 = bk.figure(
    title=f"Prices",
    width=1200, height=400,
    x_range=(ts[0], ts[-1]),
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="EUR",
)
for i, (name, price) in enumerate(zip(data.keys(), prices.T)):
    r = f1.line(times[start::skip], price[start::skip], line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f1.legend.click_policy = "hide"
bk.show(f1)

fig1 = bk.figure(
    title=f"Portfolio Value",
    width=1200, height=400,
    x_range=(ts[0], ts[-1]),
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="EUR",
)
fig1.line(ts, Vs, line_width=2, legend_label="Total V")
fig1.line(ts, Cs, line_width=1, line_dash="dashed", legend_label="Cash C")
fig1.legend.click_policy = "hide"
bk.show(fig1)

fig2 = bk.figure(
    title=f"Rewards",
    width=1200, height=400,
    x_range=(ts[0], ts[-1]),
    x_axis_type="datetime",
    x_axis_label="t",
    y_axis_label="Reward (log(V_t+1/V_t))",
)
fig2.line(ts, hist_r, line_width=2, color=Category10[10][4])

hist, edges = torch.histogram(torch.tensor(hist_r), bins=200, density=True)
x = (edges[:-1] + edges[1:]) / 2

figh = bk.figure(title="Reward Distribution", width=300, height=400)
figh.harea(y=x, x1=0, x2=hist, fill_color=Category10[10][4], fill_alpha=0.4)
figh.line(hist, x, line_color=Category10[10][4], line_width=2)

bk.show(bk.row(fig2, figh))

In [ ]:
for i, name in enumerate(data):
    print(i, name)

In [ ]:
skip = 100
Vs = torch.tensor([item["V"] for item in hist_i[::skip]])
Cs = torch.tensor([item["C"] for item in hist_i[::skip]])
ws = torch.stack([item["w"] for item in hist_i[::skip]])
ps = torch.stack([item["p"] for item in hist_i[::skip]])
vs = (ps * ws / Vs[:,None])

print(Vs.shape)
print(Cs.shape)
print(ws.shape)
print(ps.shape)
print(vs.shape)

f = bk.figure(
    title=f"Portfolio Fration",
    width=1200, height=400,
    x_range=(ts[0], ts[-1]),
    x_axis_type="datetime",
    x_axis_label=r"\(\text{t}\)",
    y_axis_label=r"\(\text{Fraction} [1]\)",
)
f.line(ts[::skip], Cs / Vs, line_width=2, line_dash="dashed", legend_label="Cash", color=Category10[10][0])
for i, name in enumerate(data):
    f.line(ts[::skip], vs[:,i], line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f.legend.click_policy = "hide"
bk.show(f)

skip = 10
f1 = bk.figure(
    title=f"Prices",
    width=1200, height=400,
    x_range=(ts[0], ts[-1]),
    x_axis_type="datetime",
    x_axis_label=r"\(\text{t}\)",
    y_axis_label=r"\(p / p_{max} [1]\)",
)
for i, (name, price) in enumerate(zip(data.keys(), prices.T)):
    r = f1.line(times[start::skip], price[start::skip] / price.max(), line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f1.legend.click_policy = "hide"
bk.show(f1)